In [2]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from sklearn.utils import class_weight

# ------------------------------------------------------------------
# 0. GPU Setup
# ------------------------------------------------------------------
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"GPU memory growth enabled for {len(gpus)} GPU(s)")
    except:
        pass

# ------------------------------------------------------------------
# 1. Load data with validation
# ------------------------------------------------------------------
import os

possible_paths = [
    '../Email Dataset/Clean_Proj_Dataset.csv',
    './Email Dataset/Clean_Proj_Dataset.csv',
    'Email Dataset/Clean_Proj_Dataset.csv',
    '../Email_Dataset/Clean_Proj_Dataset.csv',
    './Email_Dataset/Clean_Proj_Dataset.csv',
    'Clean_Proj_Dataset.csv'
]

csv_path = None
for path in possible_paths:
    if os.path.exists(path):
        csv_path = path
        break

if csv_path is None:
    raise FileNotFoundError("Could not find Clean_Proj_Dataset.csv")

print(f"Loading data from: {csv_path}")
df = pd.read_csv(csv_path)

# CRITICAL: Clean and validate data
print(f"Original data shape: {df.shape}")

# Drop rows with missing values in critical columns
df = df.dropna(subset=['clean', 'Email Type'])

# Convert labels to integers (0 or 1)
df['Email Type'] = df['Email Type'].astype(int)

# FLIP LABELS: In your dataset 1=Safe, 0=Phishing
# We need 1=SPAM/Phishing, 0=HAM/Safe for standard convention
df['Email Type'] = 1 - df['Email Type']  # Flip: 0→1, 1→0

# Verify labels are valid
assert df['Email Type'].isin([0, 1]).all(), "Labels must be 0 or 1"
print(f"After cleaning: {df.shape}")
print(f"Class distribution (0=HAM/Safe, 1=SPAM/Phishing):\n{df['Email Type'].value_counts()}")

# ------------------------------------------------------------------
# 2. Text preprocessing
# ------------------------------------------------------------------
max_words = 20_000
max_len   = 300

# Convert to strings and handle NaN/empty
clean_texts = df['clean'].fillna('').astype(str).str.strip().values

# Remove any remaining problematic entries
mask = np.array([len(text) > 0 for text in clean_texts])
clean_texts = clean_texts[mask]
labels = df['Email Type'].values[mask]

print(f"Final dataset size: {len(clean_texts)}")

# Create and adapt vectorizer
vectorizer = layers.TextVectorization(
    max_tokens=max_words,
    output_mode='int',
    output_sequence_length=max_len
)
vectorizer.adapt(clean_texts)

# ------------------------------------------------------------------
# 3. Create tf.data.Dataset
# ------------------------------------------------------------------
full_dataset = tf.data.Dataset.from_tensor_slices(
    (clean_texts, labels.astype(np.float32))
)

full_dataset = full_dataset.shuffle(
    buffer_size=len(clean_texts), seed=42, reshuffle_each_iteration=False
)

train_size = int(0.8 * len(clean_texts))
train_dataset = full_dataset.take(train_size)
val_dataset   = full_dataset.skip(train_size)

BATCH_SIZE = 64
train_dataset = train_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_dataset   = val_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# ------------------------------------------------------------------
# 4. Class weights
# ------------------------------------------------------------------
class_weights_array = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(labels),
    y=labels
)
class_weights_dict = dict(enumerate(class_weights_array))
print("Class weights:", class_weights_dict)

# ------------------------------------------------------------------
# 5. Model with STABILITY FIXES
# ------------------------------------------------------------------
model = models.Sequential([
    layers.Input(shape=(1,), dtype=tf.string),
    vectorizer,
    layers.Embedding(
        input_dim=max_words, 
        output_dim=128, 
        mask_zero=True,
        embeddings_initializer='glorot_uniform'  # Stable initialization
    ),
    layers.GlobalAveragePooling1D(),
    layers.Dense(64, activation='relu', kernel_initializer='he_normal'),
    layers.Dropout(0.5),
    layers.Dense(
        1, 
        activation='sigmoid',
        kernel_initializer='glorot_uniform',
        bias_initializer=tf.keras.initializers.Constant(-np.log((1 - labels.mean()) / labels.mean()))  # Better initial bias
    )
])

# CRITICAL: Use gradient clipping to prevent NaN
optimizer = tf.keras.optimizers.Adam(
    learning_rate=0.001,
    clipnorm=1.0  # Clip gradients to prevent explosion
)

model.compile(
    optimizer=optimizer,
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall'),
        tf.keras.metrics.AUC(name='auc')
    ]
)

model.summary()

# ------------------------------------------------------------------
# 6. Train with NaN detection
# ------------------------------------------------------------------
class NanStoppingCallback(callbacks.Callback):
    def on_batch_end(self, batch, logs=None):
        if logs and (np.isnan(logs.get('loss', 0)) or np.isinf(logs.get('loss', 0))):
            print(f"\n!!! NaN/Inf detected at batch {batch}. Stopping training.")
            self.model.stop_training = True

history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=20,
    class_weight=class_weights_dict,
    callbacks=[
        NanStoppingCallback(),
        callbacks.EarlyStopping(patience=5, restore_best_weights=True, monitor='val_loss'),
        callbacks.ReduceLROnPlateau(patience=3, factor=0.5, min_lr=1e-7, monitor='val_loss')
    ],
    verbose=1
)

# ------------------------------------------------------------------
# 7. Predict function
# ------------------------------------------------------------------
def predict(texts):
    if isinstance(texts, str):
        texts = [texts]
    texts_tensor = tf.constant(texts)
    probabilities = model(texts_tensor, training=False).numpy().flatten()
    labels_pred = ["Malicious" if p > 0.5 else "Legit" for p in probabilities]
    return labels_pred, probabilities

# ------------------------------------------------------------------
# 8. Test predictions
# ------------------------------------------------------------------
test_emails = [
    "Flight credit from cancelled trip UA-1849 expires in 8 days. Rebook here.",
    "Hey, are we still on for lunch tomorrow at 1pm?",
    "Meeting rescheduled to Friday. See you then!",
    "You won a brand new iPhone 16! Claim now before it's gone!",
    "Your credit card is now locked, click on this link to recover it"
]

labels_pred, scores = predict(test_emails)

print("\n" + "="*60)
print("PREDICTIONS")
print("="*60)
for text, label, score in zip(test_emails, labels_pred, scores):
    print(f"{label:4} ({score:.4f}) → {text}\n")

# ------------------------------------------------------------------
# 9. Save model
# ------------------------------------------------------------------
model.save('../Model/email_classifier_final.keras')
print("\nModel saved as 'email_classifier_final.keras'")

# Check if any predictions are valid
if all(np.isnan(scores)):
    print("\n⚠️  WARNING: Model is still producing NaN. Check your data!")
else:
    print("✅ Model trained successfully!")

Num GPUs Available:  1
GPU memory growth enabled for 1 GPU(s)
Loading data from: ../Email Dataset/Clean_Proj_Dataset.csv
Original data shape: (18650, 4)
After cleaning: (18628, 4)
Class distribution (0=HAM/Safe, 1=SPAM/Phishing):
Email Type
0    11321
1     7307
Name: count, dtype: int64
Final dataset size: 18628
Class weights: {0: np.float64(0.8227188410917764), 1: np.float64(1.2746681264540851)}


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization_1            │ (None, 300)            │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_1 (Embedding)         │ (None, 300, 128)       │     2,560,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_1      │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,568,321 (9.80 MB)

 Trainable params: 2,568,321 (9.80 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
233/233 ━━━━━━━━━━━━━━━━━━━━ 4s 15ms/step - accuracy: 0.8904 - auc: 0.9597 - loss: 0.2923 - precision: 0.8954 - recall: 0.8180 - val_accuracy: 0.9681 - val_auc: 0.9951 - val_loss: 0.0934 - val_precision: 0.9312 - val_recall: 0.9895 - learning_rate: 0.0010
Epoch 2/20
233/233 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step - accuracy: 0.9736 - auc: 0.9960 - loss: 0.0739 - precision: 0.9496 - recall: 0.9854 - val_accuracy: 0.9734 - val_auc: 0.9966 - val_loss: 0.0679 - val_precision: 0.9455 - val_recall: 0.9874 - learning_rate: 0.0010
Epoch 3/20
233/233 ━━━━━━━━━━━━━━━━━━━━ 4s 15ms/step - accuracy: 0.9840 - auc: 0.9983 - loss: 0.0453 - precision: 0.9657 - recall: 0.9947 - val_accuracy: 0.9750 - val_auc: 0.9966 - val_loss: 0.0612 - val_precision: 0.9505 - val_recall: 0.9859 - learning_rate: 0.0010
Epoch 4/20
233/233 ━━━━━━━━━━━━━━━━━━━━ 5s 21ms/step - accuracy: 0.9870 - auc: 0.9992 - loss: 0.0319 - precision: 0.9706 - recall: 0.9973 - val_accuracy: 0.9748 - val_auc: 0.9967 - val_loss: 0.0608 -